# CONCODE Brain — Colab GPU API (prototype)

This notebook runs the **existing FastAPI service and the real local GGUF model** from your Google Drive copy of CONCODE_BRAIN. It does not use a hosted-model substitute. It requires a Colab GPU runtime, the GGUF and fine-tuned model files in Drive, and an ngrok account for an HTTPS tunnel.

**Before running:** copy the project folder to `MyDrive/CONCODE_BRAIN`. Include `src/`, `data/`, `models/codellama-7b-concode.Q4_K_M.gguf`, `models/intent_classifier/`, and `models/code_repair/`. Do not copy `venv/` or `.env`; the notebook installs runtime dependencies and reads credentials only from Colab Secrets. The model artifacts total about 4.3 GB. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).

Add these two secrets in the Colab **Secrets** panel and enable notebook access: `CONCODE_API_KEY` (a private random value of at least 32 characters) and `NGROK_AUTHTOKEN` (from your ngrok account). Never put either value in notebook source, Drive files, or chat. Requests use `X-API-Key`.

**Prototype only:** Colab runtimes sleep, disconnect, and are not an always-on production service; the ngrok HTTPS URL changes when the tunnel/runtime restarts. The notebook verifies a real model request before exposing the tunnel. Do not configure TripVerse production with the URL until you have launched this notebook and confirmed that request succeeds.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

DRIVE_PROJECT = Path('/content/drive/MyDrive/CONCODE_BRAIN')
RUNTIME_PROJECT = Path('/content/CONCODE_BRAIN')
required = [
    DRIVE_PROJECT / 'src/service.py',
    DRIVE_PROJECT / 'data/guilds.yaml',
    DRIVE_PROJECT / 'models/codellama-7b-concode.Q4_K_M.gguf',
    DRIVE_PROJECT / 'models/intent_classifier/config.json',
    DRIVE_PROJECT / 'models/code_repair/config.json',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Copy the complete source and trained model artifacts to MyDrive/CONCODE_BRAIN. Missing: ' + ', '.join(missing))

ignore = shutil.ignore_patterns('venv', '.git', '.cache', '__pycache__', '*.pyc', '.env', '.env.*', '.DS_Store', 'outputs_intent', 'train_colab.ipynb')
shutil.copytree(DRIVE_PROJECT, RUNTIME_PROJECT, dirs_exist_ok=True, ignore=ignore)
print('Project and real model files copied to the Colab runtime.')
print('GGUF size (GB):', round((RUNTIME_PROJECT / 'models/codellama-7b-concode.Q4_K_M.gguf').stat().st_size / 1e9, 2))


In [ ]:
import os
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. Select a GPU runtime in Colab, reconnect, and rerun this cell.')
if not shutil.which('nvcc'):
    raise RuntimeError('The selected runtime has no CUDA compiler (nvcc), so a CUDA-enabled llama.cpp build is unavailable. Select a Colab GPU runtime with CUDA toolkit support.')
print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda)

base_packages = [
    'fastapi>=0.100.0', 'uvicorn[standard]>=0.22.0', 'transformers>=4.30.0',
    'sentence-transformers>=2.2.2', 'faiss-cpu>=1.7.4', 'pydantic>=2.0',
    'pyyaml>=6.0', 'python-dotenv>=1.0.0', 'numpy>=1.24.0,<2.0', 'requests>=2.31.0'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *base_packages])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'cmake', 'ninja', 'scikit-build-core'])

# Build llama.cpp with CUDA instead of silently falling back to CPU inference.
build_env = os.environ.copy()
build_env['CMAKE_ARGS'] = '-DGGML_CUDA=on'
build_env['FORCE_CMAKE'] = '1'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'llama-cpp-python'
], env=build_env)

from llama_cpp import llama_cpp
if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError('llama-cpp-python was installed without GPU offload support; stopping rather than serving the model on CPU.')
print('CUDA-enabled llama.cpp verified.')


In [ ]:
from google.colab import userdata
import time
import requests

try:
    API_KEY = userdata.get('CONCODE_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY or len(API_KEY) < 32:
    raise RuntimeError('Add a private CONCODE_API_KEY of at least 32 characters in Colab Secrets, enable notebook access, then rerun this cell.')

os.environ['CONCODE_API_KEY'] = API_KEY
os.environ['CONCODE_INFERENCE_BACKEND'] = 'local'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

log_path = Path('/content/concode-uvicorn.log')
log_file = log_path.open('w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.service:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=RUNTIME_PROJECT,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
BASE_URL = 'http://127.0.0.1:8000'
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('FastAPI exited during model startup. Inspect /content/concode-uvicorn.log; check Colab GPU memory and model artifacts.')
    try:
        if requests.get(BASE_URL + '/health', timeout=3).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError('FastAPI/model startup exceeded 15 minutes. Inspect /content/concode-uvicorn.log.')

headers = {'X-API-Key': API_KEY}
status_response = requests.get(BASE_URL + '/api/v1/status', headers=headers, timeout=15)
if not status_response.ok:
    raise RuntimeError('Authenticated status check failed with HTTP ' + str(status_response.status_code))
status = status_response.json()
generator = status['subsystems']['code_generator']
if not generator.get('loaded') or generator.get('backend') != 'local':
    raise RuntimeError('The actual local GGUF model did not load. No tunnel will be opened. Check Drive artifacts and the startup log.')
print('Authenticated API ready; local model loaded:', generator.get('model'))
print('Intent subsystem:', status['subsystems']['intent_classifier']['mode'])


In [ ]:
# Make a real inference request before opening any public tunnel.
started = time.time()
response = requests.post(
    BASE_URL + '/api/v1/cognitive',
    headers={**headers, 'Content-Type': 'application/json'},
    json={
        'query': "Write a Python function named concode_colab_check that returns 'CONCODE_REAL_MODEL_OK'.",
        'workspace_files': {},
        'mode': 'text'
    },
    timeout=300,
)
if not response.ok:
    raise RuntimeError('Real inference check failed with HTTP ' + str(response.status_code) + '. No tunnel will be opened.')
result = response.json()
if not result.get('code', '').strip() or result.get('subsystems', {}).get('generator') != 'local':
    raise RuntimeError('The request did not return output from the loaded local model. No tunnel will be opened.')
print('Real local-model inference succeeded in', round(time.time() - started, 1), 'seconds; guild:', result.get('guild'))
print('Generated output is intentionally not echoed in this notebook.')


In [ ]:
# Only after model startup and a real inference pass, open the authenticated HTTPS tunnel.
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK_TOKEN = None
if not NGROK_TOKEN:
    raise RuntimeError('Add NGROK_AUTHTOKEN as a Colab Secret and enable notebook access. Do not paste it into notebook code.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyngrok>=7.2.0'])
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(addr=8000, proto='http', bind_tls=True)
PUBLIC_URL = tunnel.public_url
if not PUBLIC_URL.startswith('https://'):
    ngrok.disconnect(PUBLIC_URL)
    raise RuntimeError('Tunnel did not provide HTTPS; refusing to publish an insecure API URL.')
print('Callable HTTPS API:', PUBLIC_URL)
print('Routes: POST /api/v1/cognitive | GET /api/v1/status | GET /api/v1/guilds | GET /health')
print('Protected routes require X-API-Key. Keep the key in your consuming server’s secret store; never put it in browser code.')
print('This URL lasts only while the Colab runtime and tunnel remain active. Do not configure TripVerse production until this URL passes an external authenticated inference request.')


## API contract and lifecycle

The existing service provides `POST /api/v1/cognitive` for code generation, `GET /api/v1/status`, `GET /api/v1/guilds`, and `GET /health`. It also has the separate `POST /api/v1/travel` contract. Protected routes require `X-API-Key`; the tunnel is HTTPS, but the URL is public, so keep the key private and rotate it if exposed.

Example request (run only in a trusted server environment; substitute your private key there):

```bash
curl -X POST "$CONCODE_BASE_URL/api/v1/cognitive" \
  -H "Content-Type: application/json" \
  -H "X-API-Key: $CONCODE_API_KEY" \
  -d '{"query":"Write a Python hello function","workspace_files":{}}'
```

Colab can stop or reclaim the runtime and ngrok’s address can change. The Drive model copy persists, but the running server does not. This is suitable for an initial integration/prototype, not a stable production endpoint.